In [1]:
# Importa pandas para manipulación y análisis de datos tabulares.
import pandas as pd
# Importa os para operaciones de sistema de archivos como creación de carpetas.
import os 
# Importa datetime para generar marcas de tiempo de los registros.
from datetime import datetime

In [2]:
# Define la ruta del archivo CSV crudo donde se almacenan los registros base.
DATA_RAW_PATH = "../../data/00_raw/traffic_data.csv"
# Define la ruta del archivo CSV procesado que contendrá transformaciones.
DATA_PROCESSED_PATH = "../../data/02_clean/traffic_data.csv"

# Declara el orden y nombres de columnas esperadas para el dataset inicial.
COLUMNS = [
    # Columna de fecha y hora de captura del registro.
    "timestamp",
    # Columna con el nombre de la avenida evaluada.
    "avenida",
    # Columna con la velocidad observada en la vía.
    "velocidad",
    # Columna con el nivel de densidad del tráfico.
    "densidad",
    # Columna con el número de detenciones o paradas.
    "detenciones"
]

In [3]:
# Define una función para crear el dataset inicial si aún no existe.
def create_dataset():
    # Crea la carpeta destino si no existe para evitar errores al guardar.
    os.makedirs(os.path.dirname(DATA_RAW_PATH), exist_ok=True)

    # Verifica si el archivo CSV crudo no existe en disco.
    if not os.path.exists(DATA_RAW_PATH):
        # Crea un DataFrame vacío con las columnas predefinidas.
        df = pd.DataFrame(columns=COLUMNS)
        # Guarda el DataFrame vacío como CSV inicial.
        df.to_csv(DATA_RAW_PATH, index=False)
        # Informa que se creó el dataset por primera vez.
        print("Dataset creado")
    else:
        # Informa que el dataset ya estaba creado previamente.
        print("Dataset ya existe")

# Ejecuta la función para asegurar que el archivo base esté disponible.
create_dataset()

Dataset ya existe


In [4]:
# Define una función para agregar un nuevo registro de tráfico al dataset crudo.
def add_record(avenida,velocidad, denssidad, detenciones):
    # Genera la marca de tiempo del momento exacto de inserción.
    timestamp = datetime.now()

    # Construye un DataFrame de una fila con los datos recibidos.
    new_data = pd.DataFrame({
        # Asigna la marca temporal capturada al campo timestamp.
        "timestamp": [timestamp],
        # Asigna el nombre de la avenida al campo avenida.
        "avenida": [avenida],
        # Asigna la velocidad observada al campo velocidad.
        "velocidad": [velocidad],
        # Asigna la densidad reportada al campo densidad.
        "densidad": [denssidad],
        # Asigna las detenciones reportadas al campo detenciones.
        "detenciones": [detenciones]
    })

    # Asegura la existencia de la carpeta destino antes de leer/escribir.
    os.makedirs(os.path.dirname(DATA_RAW_PATH), exist_ok=True)

    # Si el CSV ya existe, lo carga y concatena el nuevo registro.
    if os.path.exists(DATA_RAW_PATH):
        # Lee el dataset actual almacenado en disco.
        df =  pd.read_csv(DATA_RAW_PATH)
        # Une el dataset existente con el nuevo registro.
        df = pd.concat([df, new_data], ignore_index=True)
    else:
        # Si no existe archivo, inicializa el dataset con la nueva fila.
        df = new_data
    # Guarda el dataset actualizado en el archivo CSV crudo.
    df.to_csv(DATA_RAW_PATH, index=False)
    # Informa que el registro fue añadido correctamente.
    print("Registro agregado")    

In [5]:
# Inserta un registro de ejemplo para la avenida López Mateos.
add_record("Lopez Mateos", 25, 3, 2)
# Inserta un segundo registro de ejemplo con menor velocidad.
add_record("Lopez Mateos", 20, 3, 2)
# Inserta un tercer registro de ejemplo para simular continuidad temporal.
add_record("Lopez Mateos", 15, 3, 2)

Registro agregado
Registro agregado
Registro agregado


In [6]:
# Carga el dataset crudo desde el archivo CSV.
df = pd.read_csv(DATA_RAW_PATH)
# Imprime las primeras filas para verificar el contenido cargado.
print(df.head())

                    timestamp       avenida  velocidad  densidad  detenciones
0  2026-03-17 22:31:30.850956  Lopez Mateos       25.0         3          2.0
1  2026-03-17 22:31:30.859960  Lopez Mateos       20.0         3          2.0
2  2026-03-17 22:31:30.882579  Lopez Mateos       15.0         3          2.0
3  2026-03-17 22:42:42.487827  Lopez Mateos       25.0         3          2.0
4  2026-03-17 22:42:42.493310  Lopez Mateos       20.0         3          2.0


In [7]:
# Convierte la columna timestamp a tipo fecha-hora para análisis temporal.
df["timestamp"] = pd.to_datetime(df["timestamp"])
# Ordena el DataFrame cronológicamente por la columna timestamp.
df = df.sort_values(by ="timestamp")

# Muestra una vista previa del DataFrame ordenado.
df.head()

,timestamp,avenida,velocidad,densidad,detenciones
0,2026-03-17 22:31:30.850956,Lopez Mateos,25.0,3,2.0
1,2026-03-17 22:31:30.859960,Lopez Mateos,20.0,3,2.0
2,2026-03-17 22:31:30.882579,Lopez Mateos,15.0,3,2.0
3,2026-03-17 22:42:42.487827,Lopez Mateos,25.0,3,2.0
4,2026-03-17 22:42:42.493310,Lopez Mateos,20.0,3,2.0


In [8]:
# Define una función para normalizar una serie con min-max scaling.
def min_max_norm(series):
    # Calcula el valor mínimo de la serie.
    min_val = series.min()
    # Calcula el valor máximo de la serie.
    max_val = series.max()
    # Evita división entre cero cuando todos los valores son iguales.
    if max_val == min_val:
        # Retorna una serie de ceros conservando el mismo índice.
        return pd.Series(0.0, index=series.index)
    # Retorna la serie normalizada en el rango de 0 a 1.
    return (series - min_val) / (max_val - min_val)

# Crea la versión normalizada de la columna velocidad.
df["velocidad_norm"] = min_max_norm(df["velocidad"])
# Crea la versión normalizada de la columna densidad.
df["densidad_norm"] = min_max_norm(df["densidad"])
# Crea la versión normalizada de la columna detenciones.
df["detenciones_norm"] = min_max_norm(df["detenciones"])

# Muestra una vista previa con las nuevas columnas normalizadas.
df.head()

,timestamp,avenida,velocidad,densidad,detenciones,velocidad_norm,densidad_norm,detenciones_norm
0,2026-03-17 22:31:30.850956,Lopez Mateos,25.0,3,2.0,1.0,0.0,0.0
1,2026-03-17 22:31:30.859960,Lopez Mateos,20.0,3,2.0,0.5,0.0,0.0
2,2026-03-17 22:31:30.882579,Lopez Mateos,15.0,3,2.0,0.0,0.0,0.0
3,2026-03-17 22:42:42.487827,Lopez Mateos,25.0,3,2.0,1.0,0.0,0.0
4,2026-03-17 22:42:42.493310,Lopez Mateos,20.0,3,2.0,0.5,0.0,0.0


In [9]:
# Crea el directorio de salida procesada si no existe.
os.makedirs(os.path.dirname(DATA_PROCESSED_PATH), exist_ok=True)
# Guarda el DataFrame procesado en la ruta definida.
df.to_csv(DATA_PROCESSED_PATH, index=False)
# Confirma en consola que los datos procesados fueron almacenados.
print("Datos procesados guardados") 

Datos procesados guardados


In [10]:
# Calcula el Traffic Stability Index combinando densidad, detenciones y velocidad normalizadas.
df["TSI"] = (df["densidad_norm"] + df["detenciones_norm"]) / (df["velocidad_norm"] + 0.01)
# Muestra una vista previa con timestamp, avenida y el TSI calculado.
df[["timestamp", "avenida", "TSI"]].head()

,timestamp,avenida,TSI
0,2026-03-17 22:31:30.850956,Lopez Mateos,0.0
1,2026-03-17 22:31:30.859960,Lopez Mateos,0.0
2,2026-03-17 22:31:30.882579,Lopez Mateos,0.0
3,2026-03-17 22:42:42.487827,Lopez Mateos,0.0
4,2026-03-17 22:42:42.493310,Lopez Mateos,0.0
